# Dataset Preparation for Russian NER (factRuEval‑2016).
## This notebook contains the first stage of the project — preparing the factRuEval‑2016 dataset for training a Named Entity Recognition (NER) model based on RuModernBERT.
## Only essential preprocessing steps are included here: loading the dataset, unpacking its nested structure, building BIO labels, and creating a compact training subset.

# Why this step matters?
## The factRuEval‑2016 dataset has a non‑standard structure: samples are stored inside a nested field, and entity annotations are provided at the token level.
## To train a transformer‑based NER model correctly, the dataset must be converted into a standard HuggingFace format and enriched with:
### - BIO labels aligned with token sequences;
### - label dictionaries (id2label, label2id);
### - consistent entity type definitions (PER / ORG / LOC);
### - utility functions for text reconstruction and offset handling.
## These steps form the foundation for all subsequent experiments: tokenization, subword alignment, baseline evaluation, ModernBERT fine‑tuning, MLM pretraining, concept masking, synthetic data generation, and final comparison.

# Imports, Seeds, Device.

In [ ]:
import gc
import random
import re
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

# Loading and unpacking factRuEval‑2016.
## The dataset has a nested structure and must be unpacked into a standard HuggingFace DatasetDict.

In [ ]:
raw = load_dataset("gusevski/factrueval2016")

def unwrap_split(split_ds: Dataset) -> Dataset:
    data_list = split_ds[0]["data"]
    return Dataset.from_list(data_list)

ds = DatasetDict(
    {
        "train": unwrap_split(raw["train"]),
        "validation": unwrap_split(raw["validation"]),
        "test": unwrap_split(raw["test"]),
    }
)

ds

# Building BIO labels and dictionaries
## We extract all unique BIO tags and create id2label / label2id mappings required for token classification models.

In [ ]:
ll_labels = set()
for ex in ds["train"]:
    all_labels.update(ex["ner_tags_str"])

other_labels = sorted(l for l in all_labels if l != "O")
label_names = ["O"] + other_labels

id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
ENTITY_TYPES = ("PER", "ORG", "LOC")

def add_numeric_labels(example):
    example["ner_tags"] = [label2id[tag] for tag in example["ner_tags_str"]]
    return example

ds = ds.map(add_numeric_labels)

# Creating a compact dataset (ds_small).
## A reduced dataset is used for faster experimentation and model iteration.

train_size = 7000
val_size = 2000
test_size = 2000

ds_small = DatasetDict(
    {
        "train": ds["train"].shuffle(seed=SEED).select(range(train_size)),
        "validation": ds["validation"].shuffle(seed=SEED).select(range(val_size)),
        "test": ds["test"].shuffle(seed=SEED).select(range(test_size)),
    }
)

ds_small

Utility functions.
These functions reconstruct text from tokens, normalize whitespace, and compute character offsets. They are required for baseline models and error analysis.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "").replace("\u200b", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokens_to_text_and_offsets(tokens: List[str]) -> Tuple[str, List[Tuple[int, int]]]:
    parts = []
    offsets = []
    cur = 0
    for i, tok in enumerate(tokens):
        if i > 0:
            parts.append(" ")
            cur += 1
        start = cur
        parts.append(tok)
        cur += len(tok)
        end = cur
        offsets.append((start, end))
    text = normalize_text("".join(parts))
    return text, offsets